# 🌊 Week 3, Day 3 — June 3, 2026
## Layer Marine Species & SST Data onto Spatial Grid



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw": BASE_DIR+"/data/raw","processed": BASE_DIR+"/data/processed",
        "metadata": BASE_DIR+"/data/metadata","visualizations": BASE_DIR+"/visualizations"}
LAT_MIN,LAT_MAX,LON_MIN,LON_MAX = 5,30,65,95
YEAR_MIN,YEAR_MAX = 2015,2026

In [ ]:
df_species = pd.read_csv(DIRS['processed']+'/species_clean_final.csv')
df_climate = pd.read_csv(DIRS['processed']+'/climate_clean_v1.csv', parse_dates=['Date'])
plastic_grid = pd.read_csv(DIRS['processed']+'/plastic_grid_aggregated.csv')
print(f'species: {df_species.shape}, climate: {df_climate.shape}, plastic_grid: {plastic_grid.shape}')

## Step 1: Assign Species to Grid

In [ ]:
import numpy as np

df_species['grid_lat'] = np.floor(df_species['latitude']).astype(int)
df_species['grid_lon'] = np.floor(df_species['longitude']).astype(int)
df_species['year']     = df_species['year'].astype(int)
df_species['month']    = df_species['month'].astype(int)

species_grid = df_species.groupby(['grid_lat','grid_lon','year']).agg(
    Species_Count         = ('species_common_name',  'count'),
    Unique_Species        = ('species_common_name',  'nunique'),
    Taxonomic_Groups      = ('taxonomic_group',       lambda x: '|'.join(sorted(x.unique()))),
    Dominant_IUCN         = ('iucn_red_list_status',  lambda x: x.mode()[0] if len(x)>0 else 'DD'),
    Observation_Types     = ('observation_type',      lambda x: '|'.join(sorted(x.unique())))
).reset_index()

print(f'Species aggregated grid: {len(species_grid):,} rows')
print(f'  Unique grid cells: {species_grid[["grid_lat","grid_lon"]].drop_duplicates().shape[0]}')
print()
print('Top 5 species-richest grid cells:')
print(species_grid.nlargest(5,'Unique_Species')[['grid_lat','grid_lon','year','Unique_Species','Taxonomic_Groups']].to_string(index=False))

## Step 2: Extract Monthly SST per Location

In [ ]:
df_climate['year']  = df_climate['Date'].dt.year.astype(int)
df_climate['month'] = df_climate['Date'].dt.month.astype(int)

# Average SST and pH by year+month (global averages for the climate dataset)
sst_monthly = df_climate.groupby(['year','month']).agg(
    SST_C         = ('SST (°C)',        'mean'),
    pH            = ('pH Level',        'mean'),
    Species_Obs   = ('Species Observed','mean'),
    Heatwave_Pct  = ('Marine Heatwave', 'mean')
).reset_index()

print(f'SST monthly aggregated: {len(sst_monthly)} rows')
print()
print('SST sample:')
print(sst_monthly.head(6).round(3).to_string(index=False))

In [ ]:
# Save both grids
species_grid.to_csv(DIRS['processed']+'/species_grid_aggregated.csv', index=False)
sst_monthly.to_csv( DIRS['processed']+'/sst_monthly_aggregated.csv',  index=False)
print('Saved species_grid_aggregated.csv and sst_monthly_aggregated.csv ✅')